### 4. 动态规划算法

#### 简介

- 动态规划基本思想: 将待求问题分解成若干个子问题, 先求解子问题, 然后从这些子问题的解得到目标问题的解
- 基于动态规划的强化学习算法: 策略迭代 (policy iteration)、价值迭代 (value iteration)
- 策略迭代由两部分组成：策略评估（policy evaluation）和策略提升（policy improvement）

#### 策略迭代算法
- 策略评估 (PE) + 策略提升 (PI) 不断循环交替, 直至得到最优的策略

$$
\pi^0 \xrightarrow{\text{策略评估}} V^{\pi^0}
\xrightarrow{\text{策略提升}} \pi^1
\xrightarrow{\text{策略评估}} V^{\pi^1}
\xrightarrow{\text{策略提升}} \pi^2
\xrightarrow{\text{策略评估}} \cdots
\xrightarrow{\text{策略提升}} \pi^*
$$

##### 策略评估 PE
- 定义: 给定一个固定策略 $\pi$ , 利用贝尔曼期望方程反复更新状态价值函数直到收敛，得到该策略的价值 $V^\pi(s)$
- 核心公式: 

$$
V^{k+1}(s) = \sum_{a \in A} \pi(a \mid s) \left( r(s,a) + \gamma \sum_{s' \in S} P(s' \mid s,a)V^k(s') \right)
$$

##### 策略提升 PI
- 定义: 根据当前策略评估得到的价值函数, 重新选择每个状态下动作价值最大的动作, 从而得到更优或至少不差的新策略
- 核心公式: 

$$
\pi'(s) = \arg\max_a Q^\pi(s,a) = \arg\max_a \left\{ r(s,a) + \gamma \sum_{s'} P(s' \mid s,a)V^\pi(s') \right\}
$$


In [ ]:
import copy

class PolicyIteration:
    """ 策略迭代算法 """
    def __init__(self, env, theta, gamma):
        self.env = env # 保存悬崖漫步环境
        self.v = [0] * self.env.ncol * self.env.nrow  # 初始化价值为0
        self.pi = [[0.25, 0.25, 0.25, 0.25]
                   for i in range(self.env.ncol * self.env.nrow)]  # 初始化为均匀随机策略
        self.theta = theta  # 策略评估收敛阈值
        self.gamma = gamma  # 折扣因子

    """ 策略评估 """
    def policy_evaluation(self):
        cnt = 1                   # 记录策略评估的迭代次数
        while 1:                  # 不断迭代, 直到状态价值收敛
            max_diff = 0          # 初始化本轮最大的状态价值变化
            new_v = [0] * self.env.ncol * self.env.nrow       # 创建新一轮的状态价值列表
            for s in range(self.env.ncol * self.env.nrow):    # 遍历所有状态s
                qsa_list = []                                 # 开始计算状态s下的所有Q(s,a)价值
                for a in range(4):                            # 遍历当前状态下的4种动作
                    qsa = 0                                   # 初始化当前动作的价值Q(s,a)
                    for res in self.env.P[s][a]:              # 遍历当前状态和动作对应的所有可能转移
                        p, next_state, r, done = res          # 读取转移概率、下一状态、奖励和终止标志
                        qsa += p * (r + self.gamma * self.v[next_state] * (1 - done)) # 根据贝尔曼方程计算动作价值Q(s,a)
                        # 本章环境比较特殊,奖励和下一个状态有关,所以需要和状态转移概率相乘
                    qsa_list.append(self.pi[s][a] * qsa)
                new_v[s] = sum(qsa_list)                              # 对所有动作的加权价值求和, 得到新的V(s)
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))   # 更新本轮最大的状态价值变化
            self.v = new_v                   # 用新一轮的状态价值替换旧价值
            if max_diff < self.theta: break  # 满足收敛条件,退出评估迭代
            cnt += 1
        print("策略评估进行%d轮后完成" % cnt)

    """ 策略提升 """
    def policy_improvement(self): 
        for s in range(self.env.nrow * self.env.ncol):  # 遍历所有状态s
            qsa_list = []                               # 存储当前状态下4个动作的Q(s,a)
            for a in range(4):                          # 遍历4种动作
                qsa = 0                                 # 初始化当前动作价值Q(s,a)
                for res in self.env.P[s][a]:            # 遍历当前状态和动作对应的所有可能转移
                    p, next_state, r, done = res        # 读取转移概率、下一状态、奖励和终止标志
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done)) # 根据当前V(s)计算动作价值Q(s,a)
                qsa_list.append(qsa)     # 保存当前动作的Q值
            maxq = max(qsa_list)         # 找到当前状态下最大的动作价值
            cntq = qsa_list.count(maxq)  # 计算有几个动作得到了最大的Q值

            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list] # 将选择概率平均分配给所有最优动作
        print("策略提升完成")
        return self.pi                   # 返回改进后的策略

    """ 定义完整的策略迭代方法 """
    def policy_iteration(self): 
        while 1:
            self.policy_evaluation()            # 固定当前策略，评估其状态价值函数
            old_pi = copy.deepcopy(self.pi)     # 将列表进行深拷贝,方便接下来进行比较
            new_pi = self.policy_improvement()  # 根据当前状态价值函数改进策略
            if old_pi == new_pi: break          # 如果策略不再变化，则结束策略迭代

#### 价值迭代 (Value Iteration)
- 定义: 通过反复应用贝尔曼最优方程, 直接更新状态价值函数直到收敛, 再根据最终价值函数得到最优策略
- 核心公式: 

$$
V_{k+1}(s) = \max_a \sum_{s',r} p(s',r \mid s,a) \left[r + \gamma V_k(s')\right]
$$


In [ ]:
class ValueIteration:
    """ 价值迭代算法 """
    def __init__(self, env, theta, gamma):
        self.env = env                                # 保存悬崖漫步环境
        self.v = [0] * self.env.ncol * self.env.nrow  # 初始化价值为0
        self.theta = theta                            # 设置价值函数的收敛阈值
        self.gamma = gamma                            # 设置折扣因子
        self.pi = [None for i in range(self.env.ncol * self.env.nrow)] # 价值迭代结束后得到的策略

    def value_iteration(self):
        cnt = 0
        while 1:
            max_diff = 0                                    # 初始化本轮最大的价值变化
            new_v = [0] * self.env.ncol * self.env.nrow     # 创建新一轮状态价值列表
            for s in range(self.env.ncol * self.env.nrow):  # 遍历所有状态s
                qsa_list = []                               # 开始计算状态s下的所有Q(s,a)价值
                for a in range(4):                          # 遍历上下左右4种动作
                    qsa = 0                                 # 初始化当前动作价值Q(s,a)
                    for res in self.env.P[s][a]:            # 遍历当前状态和动作的所有可能转移
                        p, next_state, r, done = res        # 获取转移概率、下一状态、奖励和终止标志
                        qsa += p * (r + self.gamma * self.v[next_state] *
                                    (1 - done))             # 根据贝尔曼方程计算当前动作价值Q(s,a)
                    qsa_list.append(qsa)                    # 这一行和下一行代码是价值迭代和策略迭代的主要区别
                new_v[s] = max(qsa_list)                    # 选择最大的动作价值作为新状态价值V(s)
                max_diff = max(max_diff, abs(new_v[s] - self.v[s]))  # 更新本轮最大价值变化
            self.v = new_v                                  # 使用新状态价值替换旧状态价值
            if max_diff < self.theta: break                 # 满足收敛条件,退出评估迭代
            cnt += 1
        print("价值迭代一共进行%d轮" % cnt)
        self.get_policy()

    def get_policy(self):  # 根据价值函数导出一个贪婪策略
        for s in range(self.env.nrow * self.env.ncol):
            qsa_list = []
            for a in range(4):
                qsa = 0
                for res in self.env.P[s][a]:
                    p, next_state, r, done = res
                    qsa += p * (r + self.gamma * self.v[next_state] * (1 - done))
                qsa_list.append(qsa)
            maxq = max(qsa_list)
            cntq = qsa_list.count(maxq)  # 计算有几个动作得到了最大的Q值
            # 让这些动作均分概率
            self.pi[s] = [1 / cntq if q == maxq else 0 for q in qsa_list]


# env = CliffWalkingEnv()
# action_meaning = ['^', 'v', '<', '>']
# theta = 0.001
# gamma = 0.9
# agent = ValueIteration(env, theta, gamma)
# agent.value_iteration()
# print_agent(agent, action_meaning, list(range(37, 47)), [47])

# 价值迭代一共进行14轮
# 状态价值：
# -7.712 -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710
# -7.458 -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900
# -7.176 -6.862 -6.513 -6.126 -5.695 -5.217 -4.686 -4.095 -3.439 -2.710 -1.900 -1.000
# -7.458  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000
# 策略：
# ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo
# ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovo> ovoo
# ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ooo> ovoo
# ^ooo **** **** **** **** **** **** **** **** **** **** EEEE